# 04 · Fault-injection attack loop — crowbar and/or EMFI over the target UART

This is the full ChipWhisperer-style **attack → observe → classify** loop
against a real target, modeled on
[SimpleLink-FI notebook 5](https://github.com/KULeuven-COSIC/SimpleLink-FI/blob/main/notebooks/5_ChipSHOUTER-PicoEMP.ipynb).
Our target is an **Arduino Uno** running `arduino/uart_glitch_target`: send it
the command byte and it counts to 10000, then reports the count **back over
FaultyCat's own target UART**. FaultyCat voltage-glitches the Uno's 5 V rail
mid-count — so if the count that comes back isn't 10000, you know the glitch
landed.

We let `fc.GlitchController` do the bookkeeping (just like CW's): it owns the
sweep, the per-attempt classification, and the result map, leaving the loop body
to handle only the target-specific glitch-and-observe.

**Wiring** (see `arduino/README.md`): crowbar OUTPUT → Uno **5V**; FaultyCat
**GP0**→Uno **D2**; Uno **D3**→1 kΩ→FaultyCat **GP1**; common GND. *(optional)* FaultyCat **GP2**→Uno **RESET** for clean-state resets (`RST_GP`).

> Voltage glitch only — no HV. The EMFI version of this same loop is notebook 05.

In [ ]:
import time
import faultycat as fc

# --- edit for your target ---
SIM       = False              # True to dry-run without a board
CMD       = b'\xAA'           # byte that makes the target run + report
EXPECTED  = b'10000'           # the NORMAL answer (count, ASCII) == no fault
BAUD      = 9600               # target UART baud (matches the sketch)

# Engines fired each attempt — ONE loop, both optional. Enable one or BOTH:
USE_CROWBAR = True             # voltage glitch (crowbar on the 5 V rail)
USE_EMFI    = False            # EM glitch (coil over the chip) — HV! True only when set up + safe
OUTPUT        = 'lp'           # crowbar path: 'lp' or 'hp'
EMFI_WIDTH_US = 8              # EMFI pulse width (us), fixed; the swept `width` axis is the crowbar's (ns)
GAP_US        = 100            # inter-pulse gap for both engines (delay_us)

# Two ways to give a sweep vector (set_range accepts any iterable):
WIDTHS_NS = [500, 1000, 2000, 4000, 6000, 8000, 10000, 12000]  # explicit list — crowbar pulse width (ns)
PULSES    = range(1, 9)        # range(start, stop) — 1..8 GLITCH PULSES per trigger (multipulse, both engines)

# Hardware target reset (clean state each try). GP2 -> Arduino RESET. Reliable
# with the patched firmware (LOW->HIGH->hi-Z) + shifter VREF at target Vcc (5 V).
RST_GP    = 2                  # None to skip (passive brownout recovery)
RST_MS    = 10

cat = fc.connect(simulator=SIM)
cat.uart.open(baud=BAUD)
print('target says:', cat.uart.readline(timeout=1.5))   # its BOOT banner
cat.crowbar

## Your success condition

Just like in ChipWhisperer, **you** decide what counts as a hit by watching the
target. `EXPECTED` means normal; a **different count** means you injected a
fault; **nothing back** means the target hung or rebooted mid-count. Every
non-normal outcome is the glitch disturbing the computation — if it isn't the
expected count, it worked.

In [ ]:
def classify(line: bytes) -> str:
    s = line.strip()
    if s == EXPECTED:  return 'normal'    # target unfazed
    if not s:          return 'crash'     # no answer — hung
    if b'BOOT' in s:   return 'reset'     # banner ANYWHERE => the glitch rebooted the
                                          # target (read_until swallows it when the
                                          # corrupted count lost its '\n' terminator)
    return 'success'                      # corrupted count, target survived — a clean fault

## The attack loop — one loop, crowbar and/or EMFI

`GlitchController` walks the **pulses × width** grid; the loop body runs one
attempt. We send `CMD` first so the target kicks off its ~45 ms count, then fire
whichever engines are enabled — **`USE_CROWBAR` and `USE_EMFI`** — inside that
same window. Flip both on for a combined voltage + EM attack; nothing else in
the loop changes, which is the whole point of the CW model: it's engine-agnostic.

Order matters here. EMFI's HV cap is slow to charge, so we `arm()` +
`wait_for_charged()` the EMFI engine **first**, then arm the crowbar (which is
instant), then fire both back-to-back.

**Sweep vectors** — `set_range` takes any iterable:
- `width`  → an explicit **list** (`WIDTHS_NS`): the **crowbar** pulse width (ns).
  (EMFI uses the fixed `EMFI_WIDTH_US`; swap which engine the axis drives if you'd
  rather sweep EMFI instead.)
- `pulses` → **`range(1, 9)`**: 1..8 glitch pulses per trigger (multipulse, both engines).

> ⚠️ **`USE_EMFI = True` fires HIGH VOLTAGE** — coil over the chip, shield on,
> hands clear. The default is crowbar-only (safe). With `immediate` triggers the
> two `fire()` calls go out over USB ~ms apart (a sequence, both inside the count),
> not ns-aligned; for that, trigger both off one hardware trigger (D7 → GP8).

In [ ]:
gc = fc.GlitchController(['pulses', 'width'],
                        groups=['success', 'reset', 'crash', 'normal', 'error'])
gc.set_range('pulses', PULSES)           # range — glitch pulses (both engines)
gc.set_range('width', WIDTHS_NS)         # explicit list — crowbar pulse width (ns)

if USE_CROWBAR:
    cat.crowbar.trigger = 'immediate'; cat.crowbar.output = OUTPUT; cat.crowbar.delay_us = GAP_US
if USE_EMFI:
    cat.emfi.trigger = 'immediate'; cat.emfi.delay_us = GAP_US; cat.emfi.width_us = EMFI_WIDTH_US

fired = ' + '.join(e for e, on in [('crowbar', USE_CROWBAR), ('emfi', USE_EMFI)] if on)
print(f"firing: {fired or 'NOTHING — enable USE_CROWBAR and/or USE_EMFI'}")
print(f"{'pulses':>6} {'width_ns':>9}  outcome   reply")
for p in gc.glitch_values():
    if USE_CROWBAR:
        cat.crowbar.repeat = p['pulses']; cat.crowbar.width_ns = p['width']
    if USE_EMFI:
        cat.emfi.repeat = p['pulses']
    if RST_GP is not None:
        cat.target_reset(RST_GP, RST_MS)   # clean known state each try (their thorough_reset_dut)
        time.sleep(2.0)                    # let the Uno boot back to its idle loop (~1.8 s)
    if USE_EMFI:                           # charge HV BEFORE sending CMD — the multi-second
        cat.emfi.arm(); cat.emfi.wait_for_charged()   # charge would otherwise let the ~45 ms
    if USE_CROWBAR:                        # count finish before we fire (glitch lands too late)
        cat.crowbar.arm()
    cat.uart.reset_input()                 # drop any BOOT banner / stale bytes
    cat.uart.write(CMD)                    # NOW the target starts its ~45 ms count
    time.sleep(0.003)                      # let it enter the loop
    try:
        if USE_CROWBAR:
            cat.crowbar.fire()             # voltage glitch ...
        if USE_EMFI:
            cat.emfi.fire()                # ... then EM pulse (same count window)
        if USE_CROWBAR:
            cat.crowbar.disarm()
        if USE_EMFI:
            cat.emfi.disarm()
    except fc.EngineError:
        gc.add('error'); print(f"{p['pulses']:>6} {p['width']:>9}  error"); continue
    line = cat.uart.read_until(b'\n', timeout=0.3)
    res  = classify(line)
    gc.add(res)
    print(f"{p['pulses']:>6} {p['width']:>9}  {res:<8}  {line!r}")   # per-iteration result
    if RST_GP is None and res in ('crash', 'reset'):
        time.sleep(1.5)             # no reset wire: wait out the brownout reboot

gc.counts()

In [ ]:
gc.plot(x='pulses', y='width');

## Diagnostic — does each engine actually affect the target?

This is the same trick we used to prove the crowbar (10 µs → brownout → the Uno
reboots): fire **each engine on its own** and watch the target. If an engine
alone produces a non-`normal` outcome — the count corrupts, or a `BOOT`/reset
shows up — that engine is reaching the target. Run this once to confirm **EMFI
works** before you trust a full sweep. If EMFI-only comes back all `normal`, the
coil isn't coupling: reposition it over the chip, or it's simply too weak.

> ⚠️ **HV when EMFI fires.** Coil over the ATmega, shield on. Guarded with
> `RUN_DIAG = False`. EMFI does nothing unless the coil is physically over the
> chip — a glitch in the air won't reset anything.

In [ ]:
RUN_DIAG = False   # SAFETY: fires HV (EMFI) — coil over the chip, shield on

DIAG_N          = 5      # attempts per engine
DIAG_PULSES     = 4      # pulses per trigger (both engines)
DIAG_CB_WIDTH   = 8000   # crowbar pulse width (ns)
DIAG_EMFI_WIDTH = 12     # EMFI pulse width (us)

if not RUN_DIAG:
    print("Diagnostic off (RUN_DIAG=False). Set True with the coil placed + shield on.")
else:
    from collections import Counter
    cat.crowbar.trigger = 'immediate'; cat.crowbar.output = OUTPUT; cat.crowbar.delay_us = GAP_US
    cat.crowbar.width_ns = DIAG_CB_WIDTH; cat.crowbar.repeat = DIAG_PULSES
    cat.emfi.trigger = 'immediate'; cat.emfi.delay_us = GAP_US
    cat.emfi.width_us = DIAG_EMFI_WIDTH; cat.emfi.repeat = DIAG_PULSES

    for label, uc, ue in [('EMFI only', False, True), ('crowbar only', True, False), ('both', True, True)]:
        out = Counter()
        for _ in range(DIAG_N):
            if RST_GP is not None:
                cat.target_reset(RST_GP, RST_MS); time.sleep(2.0)   # clean state each try
            if ue: cat.emfi.arm(); cat.emfi.wait_for_charged()  # charge HV before CMD
            if uc: cat.crowbar.arm()
            cat.uart.reset_input(); cat.uart.write(CMD); time.sleep(0.003)
            try:
                if uc: cat.crowbar.fire()                           # voltage ...
                if ue: cat.emfi.fire()                              # ... then EM
                if uc: cat.crowbar.disarm()
                if ue: cat.emfi.disarm()
            except fc.EngineError:
                out['error'] += 1; continue
            out[classify(cat.uart.read_until(b'\n', timeout=0.3))] += 1
        hit = sum(v for k, v in out.items() if k != 'normal')
        print(f"{label:>13}: {dict(out)}   -> {'AFFECTS TARGET' if hit else 'no effect'}")

In [ ]:
cat.uart.close()
cat.close()